# Iris detection with governance


In [1]:
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")
%env PYTHONWARNINGS=ignore
%env JUPYTER_PLATFORM_DIRS=1

env: PYTHONWARNINGS=ignore
env: JUPYTER_PLATFORM_DIRS=1


In [2]:
# Import the necessary libraries

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics

from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently.presets import DataDriftPreset, DataSummaryPreset 

import shap

from fairlearn.metrics import MetricFrame, selection_rate, demographic_parity_difference
from fairlearn.reductions import DemographicParity

from feast import FeatureStore

import joblib

# Setting the warnings to be ignore once again after all the importing
import logging
warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

In [3]:
# Initialize the Feast feature store
# This connects to the pre-configured feature repository
store = FeatureStore(repo_path="Iris_Feast/feature_repo")

In [4]:
# Load entity dataframe with timestamps for historical feature retrieval
entity_df = pd.read_csv("data/entity.csv", parse_dates=['event_timestamp'])

# Retrieve historical features using Feast
# This gets features at specific points in time for training
data = store.get_historical_features(
    entity_df=entity_df,
    features=store.get_feature_service("feast_model_v1")
).to_df()

# Display first 10 rows of the dataset
print("First 10 rows of the dataset:")
data.head(10)

First 10 rows of the dataset:


,species,event_timestamp,sepal_length,sepal_width,petal_length,petal_width
0,setosa,2025-06-21 20:17:06.043098+00:00,4.8,3.4,1.6,0.2
1,setosa,2025-06-21 21:47:06.043098+00:00,4.7,3.2,1.6,0.2
2,setosa,2025-06-21 21:02:06.043098+00:00,5.4,3.4,1.7,0.2
3,setosa,2025-06-21 22:32:06.043098+00:00,4.4,3.0,1.3,0.2
4,setosa,2025-06-21 20:22:06.043098+00:00,4.8,3.0,1.4,0.1
5,setosa,2025-06-21 21:22:06.043098+00:00,4.8,3.4,1.9,0.2
6,setosa,2025-06-21 23:12:06.043098+00:00,5.1,3.8,1.6,0.2
7,setosa,2025-06-21 19:47:06.043098+00:00,5.4,3.9,1.7,0.4
8,setosa,2025-06-21 20:27:06.043098+00:00,4.3,3.0,1.1,0.1
9,setosa,2025-06-21 22:22:06.043098+00:00,5.5,3.5,1.3,0.2


## Evidently

In [5]:
# Map column types for evidently
schema = DataDefinition(
    numerical_columns=["sepal_length","sepal_width","petal_length","petal_width"],
    categorical_columns=["species"],
    )

In [6]:
# Intentionally create unseen data
new_data = data.copy()
noise_data = new_data[new_data['sepal_length']>2.5].reset_index(drop=True)
noise_data['sepal_length']=15.0
noise_data['species']='sesota'
noise_data['petal_length']=100.0
new_data = pd.concat([new_data,noise_data], ignore_index=True)
new_data

,species,event_timestamp,sepal_length,sepal_width,petal_length,petal_width
0,setosa,2025-06-21 20:17:06.043098+00:00,4.8,3.4,1.6,0.2
1,setosa,2025-06-21 21:47:06.043098+00:00,4.7,3.2,1.6,0.2
2,setosa,2025-06-21 21:02:06.043098+00:00,5.4,3.4,1.7,0.2
3,setosa,2025-06-21 22:32:06.043098+00:00,4.4,3.0,1.3,0.2
4,setosa,2025-06-21 20:22:06.043098+00:00,4.8,3.0,1.4,0.1
...,...,...,...,...,...,...
289,sesota,2025-06-22 06:42:06.043098+00:00,15.0,3.4,100.0,2.4
290,sesota,2025-06-22 04:02:06.043098+00:00,15.0,3.0,100.0,2.2
291,sesota,2025-06-22 06:12:06.043098+00:00,15.0,2.8,100.0,1.9
292,sesota,2025-06-22 04:12:06.043098+00:00,15.0,2.5,100.0,1.7


In [7]:
# Create evidently data sets, with original data as reference and new data for comparison
eval_orig_data = Dataset.from_pandas(data, data_definition=schema)
eval_new_data = Dataset.from_pandas(new_data, data_definition=schema)

In [8]:
# Run drift report
report = Report([
    DataDriftPreset(),
    DataSummaryPreset()
],
include_tests='True')

my_eval = report.run(eval_new_data, eval_orig_data)
my_eval

## Shap

In [9]:
# Split data into training and testing sets
# Using stratified split to maintain class distribution
random_state = 42
test_size = 0.4 # 40% of the data for testing
train, test = train_test_split(
    data, 
    test_size=test_size, 
    stratify=data['species'],  # Maintain class proportions
    random_state=random_state  # For reproducible results
)

# Prepare feature matrices and target vectors
# Features: sepal and petal measurements
X_train = train[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y_train = train.species

X_test = test[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y_test = test.species

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Training set size: 88
Test set size: 59


In [10]:
# Train the model
mod_dt = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)
mod_dt.fit(X_train,y_train)

# Make predictions on test set
prediction=mod_dt.predict(X_test)
accuracy_score=metrics.accuracy_score(prediction,y_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(accuracy_score))

The accuracy of the Decision Tree is 0.932


In [11]:
# Initialize SHAP JavaScript visualizations
shap.initjs()

# Create the explainer
explainer = shap.KernelExplainer(mod_dt.predict_proba, X_train)

# Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test)

# Virginica is class 2 (setosa=0, versicolor=1, virginica=2)
virginica_class_idx = 2

# Force plot for multiple instances (interactive plot showing all test samples)
shap.force_plot(explainer.expected_value[virginica_class_idx], 
                shap_values[...,virginica_class_idx], 
                X_test)


100%|██████████| 59/59 [00:00<00:00, 193.08it/s]


## Fairlearn

In [12]:
# Add random location attribute (0 and 1) to the dataset
np.random.seed(42)  # For reproducibility
data_with_location = data[["sepal_length","sepal_width","petal_length","petal_width","species"]].copy()
data_with_location['location'] = np.random.choice([0, 1], size=len(data), p=[0.5, 0.5])

print(f"\nLocation distribution:")
print(data_with_location['location'].value_counts())

# Prepare features and target
X = data_with_location.drop(["species", 'location'], axis=1)
y = data_with_location["species"]
sensitive_feature = data_with_location['location']

# Split the data
X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
    X, y, sensitive_feature, test_size=0.3, random_state=42, stratify=y
)

# Train baseline model
baseline_model = DecisionTreeClassifier(random_state=42)
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

print(f"\nBaseline Model Accuracy: {metrics.accuracy_score(y_test, baseline_pred):.4f}")


Location distribution:
location
0    78
1    69
Name: count, dtype: int64

Baseline Model Accuracy: 0.9333


In [13]:
# Create MetricFrame to analyze fairness metrics
mf_baseline = MetricFrame(
    metrics={
        'accuracy': metrics.accuracy_score,
    },
    y_true=y_test,
    y_pred=baseline_pred,
    sensitive_features=A_test
)

print("\nFairness Metrics by Location:")
print(mf_baseline.by_group)

print(f"\nOverall Metrics:")
print(mf_baseline.overall)

print(f"\nDifference in metrics between groups:")
print(mf_baseline.difference())

# Calculate demographic parity difference
dp_diff = demographic_parity_difference(y_test, baseline_pred, sensitive_features=A_test)
print(f"\nDemographic Parity Difference: {dp_diff:.4f}")


Fairness Metrics by Location:
          accuracy
location          
0             0.96
1             0.90

Overall Metrics:
accuracy    0.933333
dtype: float64

Difference in metrics between groups:
accuracy    0.06
dtype: float64

Demographic Parity Difference: 0.0000
